# TP5 - Apprentissage supervise en ligne, experts, regularisation, noyaux et normes

Ce notebook couvre les sections TP5 du mini-projet a partir de representations extraites par les CNN du projet.


## Objectif TP5

Le PDF demande d'integrer:

- une version d'**apprentissage supervise en ligne** a partir du CNN ou de sa derniere couche;
- des **algorithmes du premier ordre**: `Perceptron`, `normalized Perceptron`, `Passive-Aggressive`, `Online Subgradient Descent`;
- un scenario de **Prediction with Expert Advice**;
- de la **regularisation en ligne** `L1` et `L2`;
- une extension par **noyaux**;
- un rappel sur les **normes et normes duales**;
- une analyse de **validation / underfitting / overfitting`.

Le choix retenu ici est pragmatique: on extrait une representation cachee de `CNN1` ou `CNN2`, puis on applique les algorithmes online sur cette representation. Cela reste conforme au sujet, tout en etant plus simple a analyser qu'un entrainement online sur tous les poids du CNN.


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import optim

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

from data_loader import create_dataloaders, resolve_default_paths
from experiment_spec import EXPERIMENTAL_FRAME
from train_common import BinaryHingeLoss, build_model, select_device, set_seed, train_one_epoch
from utils import compute_regret, dual_norm, norm_l1, norm_l2, norm_linf


## Cadre experimental TP5

Le TP5 est centre ici sur la **classification binaire** `Smiling`, avec labels `{-1, +1}` et perte hinge. Les representions apprises par `CNN1` et `CNN2` servent d'entrees aux algorithmes online du premier ordre.

Cette strategie couvre bien le point **TP5-8.1** du PDF: on fait de l'apprentissage supervise en ligne a partir de la derniere couche cachee du CNN.


In [ ]:
defaults = resolve_default_paths()
device = select_device('auto')

MODEL_NAMES = ['simple', 'improved']
FEATURE_PRETRAIN_EPOCHS = 2
BATCH_SIZE = 32
IMAGE_SIZE = 64
SEED = 42

ONLINE_ETA = 0.05
PA_C = 1.0
EXPERT_ETA = 0.5
KERNEL_GAMMA = 1e-3
KERNEL_TRAIN_LIMIT = 256
LAMBDA_GRID = [0.0, 1e-4, 1e-3, 1e-2]

print('image_dir ->', defaults['image_dir'], Path(defaults['image_dir']).exists())
print('attr_file ->', defaults['attr_file'], Path(defaults['attr_file']).exists())
print('partition_file ->', defaults['partition_file'], Path(defaults['partition_file']).exists())
print('device ->', device)


## Extraction de representations CNN

On commence par entrainer rapidement chaque CNN sur la classification binaire, puis on extrait la representation cachee juste avant la couche de sortie. Cette representation sert ensuite de support a tous les algorithmes online du TP5.


In [ ]:
def get_classification_loaders(batch_size=BATCH_SIZE):
    return create_dataloaders(
        img_dir=defaults['image_dir'],
        attr_file=defaults['attr_file'],
        partition_file=defaults['partition_file'] if Path(defaults['partition_file']).exists() else None,
        target_type='classification',
        target_column=EXPERIMENTAL_FRAME['classification']['target_column'],
        classification_label_scheme=EXPERIMENTAL_FRAME['classification']['label_scheme'],
        batch_size=batch_size,
        image_size=IMAGE_SIZE,
        num_workers=0,
        seed=SEED,
        val_ratio=0.15,
        test_ratio=0.15,
        augment=False,
    )


def extract_hidden_features(model, inputs, model_name):
    if model_name == 'simple':
        x = model.pool(torch.relu(model.conv1(inputs)))
        x = model.pool(torch.relu(model.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(model.fc1(x))
        return x
    x = model.pool(torch.relu(model.bn1(model.conv1(inputs))))
    x = model.pool(torch.relu(model.bn2(model.conv2(x))))
    x = model.pool(torch.relu(model.conv3(x)))
    x = torch.flatten(x, 1)
    x = torch.relu(model.fc1(x))
    return x


def fit_cnn_backbone(model_name, epochs=FEATURE_PRETRAIN_EPOCHS):
    set_seed(SEED)
    loaders, _ = get_classification_loaders(batch_size=BATCH_SIZE)
    model = build_model(model_name, task='classification').to(device)
    criterion = BinaryHingeLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        train_one_epoch(model, loaders['train'], criterion, optimizer, device)
    model.eval()
    return model


@torch.no_grad()
def make_feature_split(model, model_name):
    loaders, sizes = get_classification_loaders(batch_size=BATCH_SIZE)
    feature_splits = {}
    for split, loader in loaders.items():
        features, labels = [], []
        for inputs, targets in loader:
            inputs = inputs.to(device)
            phi = extract_hidden_features(model, inputs, model_name)
            features.append(phi.cpu().numpy())
            labels.append(targets.numpy())
        feature_splits[split] = {
            'X': np.concatenate(features, axis=0).astype(np.float32),
            'y': np.concatenate(labels, axis=0).astype(np.int64),
        }
    return feature_splits, sizes


feature_bank = {}
for model_name in MODEL_NAMES:
    print(f'Extracting features with {model_name}')
    backbone = fit_cnn_backbone(model_name)
    feature_splits, split_sizes = make_feature_split(backbone, model_name)
    feature_bank[model_name] = {
        'splits': feature_splits,
        'sizes': split_sizes,
    }
    for split_name, payload in feature_splits.items():
        print(model_name, split_name, payload['X'].shape, payload['y'].shape)


## TP5-A - Algorithmes du premier ordre

On compare maintenant quatre algorithmes online sur les representations CNN:

- `Perceptron`;
- `normalized Perceptron`;
- `Passive-Aggressive`;
- `Online Subgradient Descent` avec perte hinge.


In [ ]:
def linear_scores(X, w, b):
    return X @ w + b


def signed_predictions(scores):
    return np.where(scores >= 0.0, 1, -1)


def accuracy_score(y_true, y_pred):
    return float(np.mean(y_true == y_pred))


def precision_recall_f1_np(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return float(precision), float(recall), float(f1)


def evaluate_linear_classifier(X, y, w, b):
    scores = linear_scores(X, w, b)
    preds = signed_predictions(scores)
    margins = y * scores
    hinge = np.maximum(0.0, 1.0 - margins)
    precision, recall, f1 = precision_recall_f1_np(y, preds)
    return {
        'loss': float(np.mean(hinge)),
        'accuracy': accuracy_score(y, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def run_online_linear_algorithm(X, y, algorithm, eta=ONLINE_ETA, C=PA_C, lambda_l1=0.0, lambda_l2=0.0):
    d = X.shape[1]
    w = np.zeros(d, dtype=np.float32)
    b = 0.0
    losses, predictions = [], []

    for x_t, y_t in zip(X, y):
        x_use = x_t.copy()
        if algorithm == 'normalized_perceptron':
            x_use = x_use / (np.linalg.norm(x_use) + 1e-12)

        score = float(np.dot(w, x_use) + b)
        pred = 1 if score >= 0.0 else -1
        margin = y_t * score
        hinge_loss = max(0.0, 1.0 - margin)
        perceptron_loss = max(0.0, -margin)

        predictions.append(pred)
        losses.append(hinge_loss if algorithm in {'passive_aggressive', 'osd'} else perceptron_loss)

        if algorithm == 'perceptron':
            if margin <= 0.0:
                w = w + y_t * x_use
                b = b + y_t
        elif algorithm == 'normalized_perceptron':
            if margin <= 0.0:
                w = w + y_t * x_use
                b = b + y_t
        elif algorithm == 'passive_aggressive':
            tau = max(0.0, 1.0 - margin) / (np.dot(x_use, x_use) + 1.0 / (2.0 * C))
            w = w + tau * y_t * x_use
            b = b + tau * y_t
        elif algorithm == 'osd':
            w = (1.0 - eta * lambda_l2) * w
            if margin < 1.0:
                w = w + eta * y_t * x_use
                b = b + eta * y_t
            if lambda_l1 > 0.0:
                threshold = eta * lambda_l1
                w = np.sign(w) * np.maximum(np.abs(w) - threshold, 0.0)
        else:
            raise ValueError(f'Unknown algorithm: {algorithm}')

    return {
        'algorithm': algorithm,
        'w': w,
        'b': b,
        'losses': np.asarray(losses, dtype=np.float32),
        'predictions': np.asarray(predictions, dtype=np.int64),
    }


def run_algorithm_family_on_features(feature_payload):
    X_train = feature_payload['splits']['train']['X']
    y_train = feature_payload['splits']['train']['y']
    X_val = feature_payload['splits']['val']['X']
    y_val = feature_payload['splits']['val']['y']
    X_test = feature_payload['splits']['test']['X']
    y_test = feature_payload['splits']['test']['y']

    algorithms = ['perceptron', 'normalized_perceptron', 'passive_aggressive', 'osd']
    rows = []
    fitted = {}
    for algorithm in algorithms:
        result = run_online_linear_algorithm(X_train, y_train, algorithm)
        fitted[algorithm] = result
        train_metrics = evaluate_linear_classifier(X_train, y_train, result['w'], result['b'])
        val_metrics = evaluate_linear_classifier(X_val, y_val, result['w'], result['b'])
        test_metrics = evaluate_linear_classifier(X_test, y_test, result['w'], result['b'])
        rows.append({
            'algorithm': algorithm,
            'train_loss': train_metrics['loss'],
            'train_f1': train_metrics['f1'],
            'val_loss': val_metrics['loss'],
            'val_f1': val_metrics['f1'],
            'test_loss': test_metrics['loss'],
            'test_f1': test_metrics['f1'],
            'test_accuracy': test_metrics['accuracy'],
        })
    return pd.DataFrame(rows), fitted


In [ ]:
tp5a_tables = {}
tp5a_models = {}

for model_name in MODEL_NAMES:
    table, fitted = run_algorithm_family_on_features(feature_bank[model_name])
    tp5a_tables[model_name] = table.sort_values('test_f1', ascending=False).reset_index(drop=True)
    tp5a_models[model_name] = fitted
    print(f'First-order algorithms on features from {model_name}')
    display(tp5a_tables[model_name])


## TP5-H - Prediction with Expert Advice

On construit ici un petit systeme d'experts dans lequel chaque expert est l'un des quatre algorithmes precedents. Le systeme combine leurs predictions avec une mise a jour multiplicative de type Hedge.


In [ ]:
def run_expert_advice(expert_results, y_true, eta=EXPERT_ETA):
    expert_names = list(expert_results.keys())
    n_experts = len(expert_names)
    weights = np.ones(n_experts, dtype=np.float64) / n_experts
    system_losses = []
    expert_losses = {name: [] for name in expert_names}

    expert_predictions = {name: expert_results[name]['predictions'] for name in expert_names}

    for t, y_t in enumerate(y_true):
        vote = 0.0
        losses_t = []
        for i, name in enumerate(expert_names):
            pred = expert_predictions[name][t]
            vote += weights[i] * pred
            loss_i = 0.0 if pred == y_t else 1.0
            expert_losses[name].append(loss_i)
            losses_t.append(loss_i)

        system_pred = 1 if vote >= 0.0 else -1
        system_losses.append(0.0 if system_pred == y_t else 1.0)
        weights *= np.exp(-eta * np.asarray(losses_t))
        weights /= weights.sum()

    cumulative_system = np.cumsum(system_losses)
    cumulative_expert = {name: np.cumsum(losses) for name, losses in expert_losses.items()}
    best_expert_loss = min(losses[-1] for losses in cumulative_expert.values())
    regret_curve = compute_regret(cumulative_system, best_expert_loss)
    return {
        'system_losses': np.asarray(system_losses),
        'cumulative_system': cumulative_system,
        'cumulative_expert': cumulative_expert,
        'best_expert_loss': best_expert_loss,
        'regret_curve': regret_curve,
    }


expert_runs = {}
for model_name in MODEL_NAMES:
    y_train = feature_bank[model_name]['splits']['train']['y']
    expert_runs[model_name] = run_expert_advice(tp5a_models[model_name], y_train)

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(14, 4), sharey=False)
for ax, model_name in zip(axes, MODEL_NAMES):
    run = expert_runs[model_name]
    ax.plot(run['cumulative_system'], label='system loss')
    for expert_name, cumulative_loss in run['cumulative_expert'].items():
        ax.plot(cumulative_loss, linestyle='--', alpha=0.7, label=expert_name)
    ax.plot(run['regret_curve'], linewidth=3, label='regret vs best expert')
    ax.set_title(f'Expert advice - {model_name}')
    ax.set_xlabel('Round')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()


## TP5-REG et TP5-VAL - Regularisation en ligne, validation, underfitting et overfitting

On etudie maintenant `Online Subgradient Descent` avec regularisation `L1` et `L2`, puis on selectionne `lambda` sur validation. Cela permet de couvrir a la fois les questions de regularisation online et celles de validation / underfitting / overfitting.


In [ ]:
def run_regularization_study(feature_payload, reg_type):
    X_train = feature_payload['splits']['train']['X']
    y_train = feature_payload['splits']['train']['y']
    X_val = feature_payload['splits']['val']['X']
    y_val = feature_payload['splits']['val']['y']
    X_test = feature_payload['splits']['test']['X']
    y_test = feature_payload['splits']['test']['y']
    rows = []

    for lambda_ in LAMBDA_GRID:
        if reg_type == 'l1':
            result = run_online_linear_algorithm(X_train, y_train, 'osd', eta=ONLINE_ETA, lambda_l1=lambda_)
        else:
            result = run_online_linear_algorithm(X_train, y_train, 'osd', eta=ONLINE_ETA, lambda_l2=lambda_)

        train_metrics = evaluate_linear_classifier(X_train, y_train, result['w'], result['b'])
        val_metrics = evaluate_linear_classifier(X_val, y_val, result['w'], result['b'])
        test_metrics = evaluate_linear_classifier(X_test, y_test, result['w'], result['b'])
        rows.append({
            'reg_type': reg_type,
            'lambda': lambda_,
            'train_loss': train_metrics['loss'],
            'train_f1': train_metrics['f1'],
            'val_loss': val_metrics['loss'],
            'val_f1': val_metrics['f1'],
            'test_loss': test_metrics['loss'],
            'test_f1': test_metrics['f1'],
            'weight_l1': norm_l1(result['w']),
            'weight_l2': norm_l2(result['w']),
            'weight_linf': norm_linf(result['w']),
        })
    return pd.DataFrame(rows)


regularization_results = {}
for model_name in MODEL_NAMES:
    l1_table = run_regularization_study(feature_bank[model_name], 'l1')
    l2_table = run_regularization_study(feature_bank[model_name], 'l2')
    regularization_results[model_name] = {'l1': l1_table, 'l2': l2_table}
    print(f'Regularization study for {model_name}')
    display(pd.concat([l1_table, l2_table], axis=0).reset_index(drop=True))


## TP5-K - Noyaux sur les representations CNN

Le sujet demande une extension par noyaux. Ici, on choisit une variante simple et interpretable: un **Perceptron kernelise** applique non pas aux images brutes, mais aux representations extraites par le CNN. Cela correspond au cas **TP5-K.3** du PDF.


In [ ]:
def rbf_kernel(X1, X2, gamma=KERNEL_GAMMA):
    X1_norm = np.sum(X1 ** 2, axis=1, keepdims=True)
    X2_norm = np.sum(X2 ** 2, axis=1, keepdims=True).T
    distances = X1_norm + X2_norm - 2.0 * X1 @ X2.T
    return np.exp(-gamma * distances)


def fit_kernel_perceptron(X_train, y_train, gamma=KERNEL_GAMMA):
    n = len(y_train)
    alpha = np.zeros(n, dtype=np.float32)
    K = rbf_kernel(X_train, X_train, gamma=gamma)
    for i in range(n):
        score = np.sum(alpha * y_train * K[:, i])
        pred = 1 if score >= 0.0 else -1
        if pred != y_train[i]:
            alpha[i] += 1.0
    return {'alpha': alpha, 'X_train': X_train, 'y_train': y_train, 'gamma': gamma}


def predict_kernel_perceptron(model, X):
    K = rbf_kernel(model['X_train'], X, gamma=model['gamma'])
    scores = np.sum((model['alpha'] * model['y_train'])[:, None] * K, axis=0)
    return np.where(scores >= 0.0, 1, -1)


kernel_rows = []
for model_name in MODEL_NAMES:
    X_train = feature_bank[model_name]['splits']['train']['X'][:KERNEL_TRAIN_LIMIT]
    y_train = feature_bank[model_name]['splits']['train']['y'][:KERNEL_TRAIN_LIMIT]
    X_val = feature_bank[model_name]['splits']['val']['X']
    y_val = feature_bank[model_name]['splits']['val']['y']

    kernel_model = fit_kernel_perceptron(X_train, y_train)
    val_pred = predict_kernel_perceptron(kernel_model, X_val)
    val_acc = accuracy_score(y_val, val_pred)
    _, _, val_f1 = precision_recall_f1_np(y_val, val_pred)
    kernel_rows.append({
        'model': model_name,
        'kernel_train_limit': KERNEL_TRAIN_LIMIT,
        'gamma': KERNEL_GAMMA,
        'val_accuracy': val_acc,
        'val_f1': val_f1,
    })

kernel_table = pd.DataFrame(kernel_rows)
display(kernel_table)


## TP5-ND - Normes et normes duales

Les normes usuelles importantes ici sont:

- `L1`: favorise la parcimonie;
- `L2`: penalise l'energie globale du vecteur de poids;
- `L_infinity`: controle la plus grande composante en valeur absolue.

Les normes duales associees sont:

- duale de `L1` = `L_infinity`;
- duale de `L2` = `L2`;
- duale de `L_infinity` = `L1`.

Dans les algorithmes online, ces normes apparaissent dans les termes de regularisation et dans les bornes de regret.


In [ ]:
norm_rows = []
for model_name in MODEL_NAMES:
    for algorithm_name, payload in tp5a_models[model_name].items():
        w = payload['w']
        norm_rows.append({
            'model': model_name,
            'algorithm': algorithm_name,
            'l1': norm_l1(w),
            'l2': norm_l2(w),
            'linf': norm_linf(w),
            'dual_l1': dual_norm(w, primal='l1'),
            'dual_l2': dual_norm(w, primal='l2'),
            'dual_linf': dual_norm(w, primal='linf'),
        })
norm_table = pd.DataFrame(norm_rows)
display(norm_table)


## Lecture underfitting / overfitting

Une fois les tableaux executes, il faut lire les tendances de la facon suivante:

- si `train_f1` et `val_f1` sont tous deux faibles, on est plutot en **underfitting**;
- si `train_f1` est eleve mais `val_f1` degrade, on suspecte de l'**overfitting**;
- si augmenter `lambda` reduit l'ecart train/validation sans trop casser les performances, la regularisation aide;
- si `lambda` devient trop grand et degrade train et validation, on regularise trop fort.


## Synthese TP5

Ce notebook couvre les items du PDF de la facon suivante:

- **TP5-8.1**: apprentissage supervise en ligne sur les representations de la derniere couche cachee du CNN;
- **TP5-A.1**: `Perceptron`;
- **TP5-A.2**: `normalized Perceptron`;
- **TP5-A.3**: `Passive-Aggressive`;
- **TP5-A.4**: `Online Subgradient Descent`;
- **TP5-H.1 a TP5-H.3**: systeme d'experts, perte du systeme, perte du meilleur expert, regret;
- **TP5-REG.1 a TP5-REG.3**: regularisation `L1`, `L2`, effet de `lambda`;
- **TP5-K.3**: noyau applique aux representations extraites par le CNN;
- **TP5-ND.1 a TP5-ND.3**: normes, normes duales, role dans regularisation et regret;
- **TP5-VAL.1 a TP5-VAL.4**: validation simple, diagnostic underfitting / overfitting, correction par regularisation.

Le notebook reste coherent avec tout le projet: meme dataset, memes deux CNN, meme tache binaire, puis extension naturelle vers l'apprentissage online du premier ordre.
